# Random Forest


O objetivo do projeto é prever a pontuação dos vinhos usando o algoritmo de Regressão de Random Forest para classificação multiclasse a partir de uma base de dados de avaliações de vinhos.
A base contém algumas características químicas do vinho que influemciam na sua qualidade.

**Caracaterísticas químicas do vinho:**
- Características dos Vinhos (Features)
- Fixed Acidity: Acidez fixa do vinho.
- Volatile Acidity: Acidez volátil do vinho.
- Citric Acid: Quantidade de ácido cítrico no vinho.
- Residual Sugar: Açúcar residual presente no vinho.
- Chlorides: Nível de cloretos no vinho.
- Free Sulfur Dioxide: Dióxido de enxofre livre no vinho.
- Total Sulfur Dioxide: Quantidade total de dióxido de enxofre no vinho.
- Density: Densidade do vinho.
- pH: Nível de pH do vinho.
- Sulphates: Quantidade de sulfatos no vinho.
- Alcohol: Teor alcoólico do vinho.

**Qualidade do Vinho (Variável de Saída,Target):**

Quality: Pontuação do vinho baseada em dados sensoriais, variando de 0 a 10.


In [147]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots

In [295]:
df = pd.read_csv("winequality-red.csv", delimiter=',')

df.head(10)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5
6,7.9,0.60,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5
7,7.3,0.65,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7
8,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7
9,7.5,0.50,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.80,10.5,5


# Pré processamento dos dados.

## Verifcação da base 
Todas as variáveis estão no formato adequado, não há dados ausentes ou nulos

In [168]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [169]:
print(f"\nCampos com valores nulos:\n{df.isnull().sum()}")


Campos com valores nulos:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64


# 2 - Realize a segunda e terceita etapa de pré processamento dos dados.

A) Utilize a função describe para identificarmos outliers e verificarmos a distribuição dos dados.

B) Verifique o balanceamento da váriavel Target.

C)  Plote o gráfico ou a tabela e indique as variáveis que te parecem mais "fortes" na correlação para nosso modelo.

D) Crie um novo dataframe apenas com as váriaveis que parecem ter maior correlação com a target. (Negativa ou positiva)


## Verificação de Outliers
fixed acidity: valor máximo superior ao limite 14

In [170]:
df['fixed acidity'].describe()

count    1599.000000
mean        8.319637
std         1.741096
min         4.600000
25%         7.100000
50%         7.900000
75%         9.200000
max        15.900000
Name: fixed acidity, dtype: float64

In [296]:
df = df[df['fixed acidity'] <= 14]
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000
mean,8.284538,0.528143,0.269510,2.532715,0.087463,15.901948,46.546826,0.996723,3.312872,0.657555,10.421213,5.634821
std,1.673064,0.178907,0.193986,1.405646,0.047178,10.474248,32.937178,0.001858,0.152680,0.169678,1.059742,0.806745
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996720,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997800,3.400000,0.730000,11.100000,6.000000
max,14.000000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.000000,8.000000


mean + 2*std para agilizar a análise
a ideia é que 2var para além da média não deve ser muito distante de 75 nem max, se for, algo estranho está aocntecendo e vale avaliar mais
uma análise um pouco subjetiva, mas para uma avaliação rápida, pode ajudar


as variáveis que me chamarm a atenção foram:
volatlie acidity
residual sugar
clorides
total sulfur dioxide
vamos olhar o histograma de cada um

In [297]:
qtd_base_original = df['quality'].count()
print(qtd_base_original)

1591


In [218]:
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000
mean,8.127508,0.520498,0.243534,2.423315,0.076219,16.017241,45.768809,0.998401,3.330486,0.639961,10.414381,5.655172
std,1.541770,0.164472,0.171793,1.160120,0.011843,9.873618,30.540520,0.003666,0.143907,0.137183,1.046100,0.785294
min,4.600000,0.120000,0.000000,0.900000,0.045000,1.000000,6.000000,0.990000,2.900000,0.330000,8.700000,3.000000
25%,7.100000,0.390000,0.080000,1.900000,0.069000,8.000000,23.000000,1.000000,3.200000,0.550000,9.500000,5.000000
50%,7.800000,0.520000,0.240000,2.200000,0.077000,14.000000,38.000000,1.000000,3.300000,0.610000,10.100000,6.000000
75%,9.000000,0.630000,0.390000,2.500000,0.084000,22.000000,61.000000,1.000000,3.400000,0.710000,11.000000,6.000000
max,13.300000,1.000000,0.600000,15.500000,0.100000,57.000000,149.000000,1.000000,3.900000,1.620000,14.000000,8.000000


In [173]:
for campo in df.columns:
    fig = make_subplots(rows=1, cols=2, subplot_titles=[f'Histograma de {campo}', f'Box Plot de {campo}'])
    fig.add_trace(
        px.histogram(df, x=campo, histnorm="percent", nbins=60).data[0],
        row=1, col=1
    )
    fig.add_trace(
        px.box(df, y=campo).data[0],
        row=1, col=2
    )
    fig.update_layout(title_text=f'{campo}', showlegend=False)
    fig.show()
    

In [298]:
campos = ["75%", "IQR", "Max"]
dsup = pd.DataFrame(columns=campos)
i = 0
for campo in df:
    dsup.loc[campo] = [
        df[campo].quantile(0.75),
        (df[campo].mean()+1.5*(df[campo].quantile(0.75)-df[campo].quantile(0.25))),
        df[campo].max()
    ]
    i += 1
print(dsup)

                          75%         IQR        Max
fixed acidity          9.2000   11.434538   14.00000
volatile acidity       0.6400    0.903143    1.58000
citric acid            0.4200    0.764510    1.00000
residual sugar         2.6000    3.582715   15.50000
chlorides              0.0900    0.117463    0.61100
free sulfur dioxide   21.0000   36.901948   72.00000
total sulfur dioxide  62.0000  106.546826  289.00000
density                0.9978    1.000023    1.00369
pH                     3.4000    3.597872    4.01000
sulphates              0.7300    0.927555    2.00000
alcohol               11.1000   12.821213   14.00000
quality                6.0000    7.134821    8.00000


In [299]:
dsup_percent = pd.DataFrame(columns=campos)
for campo in df:
    dsup_percent.loc[campo] = (dsup.loc[campo]/dsup.loc[campo].max()*100).round(2)
print(dsup_percent)

                        75%    IQR    Max
fixed acidity         65.71  81.68  100.0
volatile acidity      40.51  57.16  100.0
citric acid           42.00  76.45  100.0
residual sugar        16.77  23.11  100.0
chlorides             14.73  19.22  100.0
free sulfur dioxide   29.17  51.25  100.0
total sulfur dioxide  21.45  36.87  100.0
density               99.41  99.63  100.0
pH                    84.79  89.72  100.0
sulphates             36.50  46.38  100.0
alcohol               79.29  91.58  100.0
quality               75.00  89.19  100.0


In [300]:
px.bar(dsup_percent[["Max", "IQR", "75%"]], barmode="overlay", opacity=0.75, title="Distribuição Percentual").show()

importante destacar que essa comparação não fala da quantidade dos dados, mas o quanto cada um desses valores representa do máximo. isso serve para entendermos em que medida a amostra tem outliers, uma análise prelimnar.
por exemplo, no caso do ph, o limiar geralmente usado para determinar se um valor é outrlier ou nao, o iqr está muito próximo do valor de max. isso indica que os valores superiores a iqr não estão muito fora da curva, sua exclusão não parece obrigatoria.
já no caso do residual sugar, vemos que a distancia entre o maximo medido e o IQR é muito grande, então esse valor está muito fora da curva.


In [ ]:
for campo in df:
     dsup_percent['count_IQR'].loc[campo] = (df[df[campo] >= dsup.loc[campo]['IQR']].count()[0]/qtd_base_original*100).round(2)
print(dsup_percent)

                        75%    IQR    Max  count_IQR
fixed acidity         65.71  81.68  100.0       6.35
volatile acidity      40.51  57.16  100.0       2.89
citric acid           42.00  76.45  100.0       0.19
residual sugar        16.77  23.11  100.0       9.99
chlorides             14.73  19.22  100.0       7.17
free sulfur dioxide   29.17  51.25  100.0       4.15
total sulfur dioxide  21.45  36.87  100.0       6.22
density               99.41  99.63  100.0       4.02
pH                    84.79  89.72  100.0       3.46
sulphates             36.50  46.38  100.0       5.59
alcohol               79.29  91.58  100.0       2.33
quality               75.00  89.19  100.0       1.13


C:\Users\marina.freitas\AppData\Local\Temp\ipykernel_17128\1305249407.py:2: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\marina.freitas\AppData\Local\Temp\ipykernel_17128\1305249407.py:2: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step a

In [334]:
print(df[df[campo] >= dsup.loc[campo]['IQR']].count()[0]/qtd_base_original*100)

1.1313639220615965


C:\Users\marina.freitas\AppData\Local\Temp\ipykernel_17128\1130705328.py:1: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



individualmente, os valores não são alto, nenhum supera 10%. porém, se todos forem elminados, a base é reduzida a 1.13% do seu tamanho. 
por esse motivo, vou escolher substituir todos pela sua média, ao invés de elimiar.

df = df[df['volatile acidity']<=1] -> substituir pela média

In [ ]:
df = df[df['volatile acidity']<=1]
df = df[df['citric acid']<=0.6]
df = df[df['chlorides']<=0.1]
df = df[df['chlorides']>=0.045]
df = df[df['free sulfur dioxide']<=60]
df = df[df['total sulfur dioxide']<=150]

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000
mean,8.127508,0.520498,0.286994,2.423315,0.076219,16.017241,45.768809,0.996569,3.328440,0.639961,10.414381,5.655172
std,1.541770,0.164472,0.171490,1.160120,0.011843,9.873618,30.540520,0.001728,0.141143,0.137183,1.046100,0.785294
min,4.600000,0.120000,0.010000,0.900000,0.045000,1.000000,6.000000,0.990070,2.880000,0.330000,8.700000,3.000000
25%,7.100000,0.390000,0.130000,1.900000,0.069000,8.000000,23.000000,0.995500,3.230000,0.550000,9.500000,5.000000
50%,7.800000,0.520000,0.290000,2.200000,0.077000,14.000000,38.000000,0.996600,3.330000,0.610000,10.100000,6.000000
75%,9.000000,0.630000,0.440000,2.500000,0.084000,22.000000,61.000000,0.997600,3.410000,0.710000,11.000000,6.000000
max,13.300000,1.000000,0.600000,15.500000,0.100000,57.000000,149.000000,1.002600,3.900000,1.620000,14.000000,8.000000


os outliers de densidade podem ser resolvidos apenas mudando o arredondamento, dado que a diferença está na casa decimal. supondo que a unidade é g/l  

In [ ]:
qdt_base_processada = df['quality'].count()
print(qdt_base_processada/qtd_base_original)

0.8020113136392206


a base após os dados alterados representa 80% dos dados originais, então tá ok essa limpeza.
densidade e ph não tem necessidade de serem excluidos, apenas arredondados

In [178]:
df['density'] = df['density'].round(2)
df['pH'] = df['pH'].round(1)

In [181]:
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000,1276.000000
mean,8.127508,0.520498,0.243534,2.423315,0.076219,16.017241,45.768809,0.998401,3.330486,0.639961,10.414381,5.655172
std,1.541770,0.164472,0.171793,1.160120,0.011843,9.873618,30.540520,0.003666,0.143907,0.137183,1.046100,0.785294
min,4.600000,0.120000,0.000000,0.900000,0.045000,1.000000,6.000000,0.990000,2.900000,0.330000,8.700000,3.000000
25%,7.100000,0.390000,0.080000,1.900000,0.069000,8.000000,23.000000,1.000000,3.200000,0.550000,9.500000,5.000000
50%,7.800000,0.520000,0.240000,2.200000,0.077000,14.000000,38.000000,1.000000,3.300000,0.610000,10.100000,6.000000
75%,9.000000,0.630000,0.390000,2.500000,0.084000,22.000000,61.000000,1.000000,3.400000,0.710000,11.000000,6.000000
max,13.300000,1.000000,0.600000,15.500000,0.100000,57.000000,149.000000,1.000000,3.900000,1.620000,14.000000,8.000000


# 3 - Preparação Final dos Dados

A) Separe a base em X(Features) e Y(Target)

B) Separe a base em treino e teste.


In [ ]:
#seu código aqui

# 4 - Modelagem

A) Inicie e treine o modelo de Random Forest

B) Aplique a base de teste o modelo.


In [ ]:
#seu código aqui

# 5 - Avaliação

A) Avalie as principais métricas da Claissificação e traga insights acerca do resultado, interprete os valores achados.

B) Você nota que o modelo teve dificuldade para prever alguma classe? Se sim, acredita que tenha relação com o balanceamento dos dados? Explique.


In [ ]:
#seu código aqui

# 5 - Melhorando os Hyperparametros

A) Defina o Grid de parametros que você quer testar

B) Inicie e Treine um novo modelo utilizando o random search.

C) Avalie os resultados do modelo.

D) Você identificou melhorias no modelo após aplicar o random search? Justifique.


ps. Essa parte da atividade demorará um pouco para rodar!

In [ ]:
#seu código aqui

# 6 - Chegando a perfeição

Baseado em tudo que você já aprendeu até agora, quais outras técnicas você acredita que poderiam ser aplicadas ao modelo para melhorar ainda mais suas previsões?